# Sand & excavation pit detection — Karnal, Haryana (Sentinel only)

This notebook finds mining, sand and brick-kiln pits — including small ones — using only free satellite data.

**How it works**

1. **Earth Engine** builds Sentinel features for the last few months and the same months a year earlier:
   Sentinel-2 (optical, 10 m), Sentinel-1 (radar), Dynamic World (Google's AI land cover),
   building maps, and AlphaEarth AI embeddings (turned into "how much does this look like a pit / building / water / crops").
2. The features are **downloaded** as one GeoTIFF (resumable).
3. Everything else runs **on your computer at full 10 m detail**:
   - *context* features — each pixel compared with its surroundings, like a CNN would see it;
   - *sub-pixel* water / bare-soil / vegetation / building fractions, so pits smaller than a pixel still leave a trace;
   - automatic training examples + **your own points**, validated on 5 km blocks the model never saw;
   - a gradient-boosting model, pit **objects** (hysteresis outlines, size / shape / building filters);
   - optional **super-resolution to 2.5 m** (SEN2SR) for close-up views and refined outlines.
4. **Visualise**: input panels, maps, accuracy tables, a gallery of pits (10 m vs 2.5 m), the history of any pit, statistics.
5. **Export**: `pits.gpkg`, `pits.geojson`, `pits.kml` (Google Earth), `pits.csv`, `pit_probability.tif`.

**Run order:** top to bottom. Start with `AREA = 'test'` (a 14 × 13 km box along the Yamuna — a few minutes),
then switch to `AREA = 'district'`. To improve results, save a few dozen points in section 3 and re-run from section 6.

**Limits (honest):** Sentinel-2 pixels are 10 m. Pits of about 3 pixels (≈ 300 m², e.g. 15 × 20 m) are the smallest this can
detect reliably; smaller ones only show if they hold water or contrast strongly. Super-resolved 2.5 m images are an AI
estimate — great for looking and for tightening outlines, not a substitute for real high-resolution imagery.

In [ ]:
# Installs (Colab / fresh environments). Safe to re-run.
%pip install -q earthengine-api geemap rasterio scikit-learn scikit-image geopandas pyproj tqdm
# Optional, for 2.5 m super-resolution (section 10):
# %pip install -q sen2sr mlstac torch

In [ ]:
import base64, io, json, math, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import ListedColormap
import rasterio
from rasterio import features as rio_features
from rasterio.transform import Affine
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import Window
from pyproj import Transformer
from scipy import ndimage
from scipy.spatial import cKDTree
from shapely.geometry import shape
from shapely.ops import unary_union
from skimage import measure
from skimage.filters import apply_hysteresis_threshold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from tqdm.auto import tqdm
from PIL import Image
from IPython.display import display

warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty windows at the area edge

## Settings

In [ ]:
# --- Earth Engine --------------------------------------------------------------
PROJECT_ID = 'your-cloud-project-id'     # the Google Cloud project you use with Earth Engine

# --- Area ------------------------------------------------------------------------
STATE = 'Haryana'
DISTRICT_NAMES = ['Karnal']
AREA = 'test'     # 'test'     = 14 x 13 km along the Yamuna east of Karnal city (quick first run)
                  # 'district' = the whole district (~1-2 GB download, roughly 20-40 minutes)
TEST_BOX = [77.02, 29.62, 77.16, 29.74]   # west, south, east, north: pits, villages and fields

# --- Time ------------------------------------------------------------------------
END_DATE = None   # None = today, or e.g. '2026-09-29' to freeze the window
MONTHS_BACK = 3   # compared with the same months one year earlier

# --- Grid ------------------------------------------------------------------------
CRS = 'EPSG:32643'   # UTM zone 43N (metres), covers Haryana
SCALE = 10           # Sentinel-2 pixel size (m)

# --- Earth Engine layers -----------------------------------------------------------
CLOUD_SCORE_MIN = 0.6
BUILT_PRESENCE, BUILT_DW, BUILT_VV = 0.5, 0.35, -6    # thresholds for the building "votes"
RIVER_MIN_DISCHARGE, RIVERBED_BUFFER = 50, 1500       # rivers >= 50 m3/s; 1.5 km either side

# --- Automatic training examples -----------------------------------------------------
NEVER_GREEN_NDVI, GREEN_NDVI = 0.25, 0.45
WET_NDWI, PERMANENT_WATER_NDWI = 0.0, 0.0
PIT_MAX_VV = -11                   # dB: clean pit examples are radar-dark
SAMPLES = {'pit': 3000, 'lookalike': 1500, 'water': 800,
           'buildings': 1500, 'crops': 1500, 'dry_bare': 1000}
USE_AUTO_LABELS = True             # False = learn only from your points (once you have ~150+ of each)
USER_WEIGHT = 5                    # each of your points counts this many times

# --- Validation ----------------------------------------------------------------------
BLOCK_PX = 500                     # 5 km blocks: whole blocks go to training or to validation
VALIDATION_PERCENT = 25

# --- Pit outlines --------------------------------------------------------------------
PROB_HIGH = 'auto'                 # 'auto' = best F1 against your points (0.6 until you have enough), or a number
MIN_PIT_PIXELS = 3                 # 0.03 ha. Pits under SMALL_PIT_PIXELS are kept only if
SMALL_PIT_PIXELS = 10              # their most confident pixel reaches SMALL_PIT_MIN_PROB
SMALL_PIT_MIN_PROB = 0.75
MIN_COMPACTNESS = 0.2              # 4*pi*area/perimeter^2: drops canals, roads, riverbank strips
MAX_TINY_ELONGATION = 2.5          # tiny pits longer than 2.5 x their width are tracks / drains
MAX_SMALL_ELONGATION = 4           # pits under 0.5 ha longer than 4 x their width are too
LINEAR_MAX_HALF_WIDTH = 3          # px: strips up to ~60 m wide...
LINEAR_MIN_LENGTH = 30             # px: ...and at least 300 m long are canals, drains or roads
VILLAGE_POND_BUILT = 0.15          # water bodies with >= 15 % buildings within ~100 m = likely village pond
MAX_BUILT_SHARE = 0.5              # drops outlines that are mostly building evidence

# --- Extras ---------------------------------------------------------------------------
USE_SUPER_RESOLUTION = True        # 2.5 m views + refined outlines (needs sen2sr, mlstac, torch)
GALLERY_SIZE = 24

# --- Derived (no need to edit) ----------------------------------------------------------
end_ts = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp(date.today())
START = (end_ts - pd.DateOffset(months=MONTHS_BACK)).strftime('%Y-%m-%d')
END = end_ts.strftime('%Y-%m-%d')
TAG = f"{DISTRICT_NAMES[0].lower()}_{AREA}_{START.replace('-', '')}_{END.replace('-', '')}"
OUT_DIR = Path('pit_outputs') / TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
STACK_PATH = OUT_DIR / 'stack.tif'
PROB_PATH = OUT_DIR / 'pit_probability.tif'
LABELS_PATH = Path('pit_outputs') / 'labels.geojson'     # your points, shared by all runs
NODATA = -32768

# Bands downloaded from Earth Engine, stored as int16 = value x factor.
BANDS = {
    'B2': 1e4, 'B3': 1e4, 'B4': 1e4, 'B8': 1e4, 'B11': 1e4, 'B12': 1e4,        # median reflectance
    'NDVI_p10': 1e4, 'NDVI_p90': 1e4,                                           # vegetation
    'NDWI_p10': 1e4, 'NDWI_p50': 1e4, 'NDWI_p90': 1e4,                          # water
    'BSI_p50': 1e4,                                                             # bare soil
    'NDVI_prev_p90': 1e4, 'NDWI_prev_p50': 1e4,                                 # a year earlier
    'VV_p10': 100, 'VV_p50': 100, 'VH_p50': 100,                                # radar (dB)
    'dw_water': 1e4, 'dw_crops': 1e4, 'dw_built': 1e4, 'dw_bare': 1e4,          # Dynamic World
    'built_presence': 1e4, 'built_votes': 1,                                    # buildings
    'sim_pit': 1e4, 'sim_buildings': 1e4, 'sim_water': 1e4, 'sim_crops': 1e4,   # AlphaEarth
    'riverbed': 1,
}
BAND_NAMES = list(BANDS)
print(f'{AREA} area, {START} to {END}; outputs in {OUT_DIR}/')

## 1. Connect to Earth Engine
The first time, this opens a sign-in link. `PROJECT_ID` is the Cloud project you registered for Earth Engine.

In [ ]:
import ee
import geemap
import ipywidgets as widgets

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
ee.data.setDeadline(240_000)      # a request that hangs fails after 4 minutes and is retried
print('Earth Engine ready.')

## 2. Sentinel features in Earth Engine
Same knowledge as the Code Editor script: cloud-free Sentinel-2 statistics, radar, Dynamic World,
four building sources that "vote", the Yamuna riverbed zone, and AlphaEarth similarity to typical pits / buildings / water / crops.

In [ ]:
districts = (ee.FeatureCollection('FAO/GAUL/2015/level2')
             .filter(ee.Filter.eq('ADM1_NAME', STATE))
             .filter(ee.Filter.inList('ADM2_NAME', DISTRICT_NAMES)))
if districts.size().getInfo() == 0:
    names = (ee.FeatureCollection('FAO/GAUL/2015/level2')
             .filter(ee.Filter.eq('ADM1_NAME', STATE)).aggregate_array('ADM2_NAME').getInfo())
    raise ValueError(f'{DISTRICT_NAMES} not found in {STATE}. Names available: {names}')
test_box = ee.Geometry.Rectangle(TEST_BOX)
aoi = test_box if AREA == 'test' else districts.geometry()
start, end = ee.Date(START), ee.Date(END)

# --- Sentinel-2, cloud-masked with Cloud Score+ ---
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

def sentinel2(t0, t1):
    def prepare(img):
        sr = img.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).divide(10000)
        bsi = sr.expression('((S1 + R) - (N + B)) / ((S1 + R) + (N + B))', {
            'S1': sr.select('B11'), 'R': sr.select('B4'), 'N': sr.select('B8'), 'B': sr.select('B2')})
        return (sr.addBands(sr.normalizedDifference(['B8', 'B4']).rename('NDVI'))
                  .addBands(sr.normalizedDifference(['B3', 'B8']).rename('NDWI'))     # 10 m bands only
                  .addBands(bsi.rename('BSI'))
                  .updateMask(img.select('cs_cdf').gte(CLOUD_SCORE_MIN))
                  .set('system:time_start', img.get('system:time_start')))
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi).filterDate(t0, t1)
            .linkCollection(cs_plus, ['cs_cdf'])
            .map(prepare))

s2 = sentinel2(start, end)
median = s2.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).median()
stats = s2.select(['NDVI', 'NDWI', 'BSI']).reduce(ee.Reducer.percentile([10, 50, 90]))
previous = (sentinel2(start.advance(-1, 'year'), end.advance(-1, 'year'))
            .select(['NDVI', 'NDWI']).reduce(ee.Reducer.percentile([50, 90]))
            .regexpRename('_p', '_prev_p'))

# --- Sentinel-1 radar ---
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate(start, end)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH'])
      .map(lambda img: img.updateMask(img.select('VV').gt(-30))))
radar = s1.reduce(ee.Reducer.percentile([10, 50, 90]))

# --- Dynamic World: share of the window each pixel was water / crops / built / bare ---
DW = ['water', 'crops', 'built', 'bare']
dynamic_world = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(aoi).filterDate(start, end)
                 .select(DW).mean().rename(['dw_' + c for c in DW]))

# --- Building knowledge: four sources vote "building here" ---
obt = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1').filterBounds(aoi)
building_presence = (obt.filter(ee.Filter.eq('system:time_start', obt.aggregate_max('system:time_start')))
                     .mosaic().select('building_presence').unmask(0).rename('built_presence'))
worldcover_built = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').eq(50)
built_votes = ee.Image.cat(
    worldcover_built.unmask(0),
    building_presence.gt(BUILT_PRESENCE),
    dynamic_world.select('dw_built').gt(BUILT_DW).unmask(0),
    radar.select('VV_p50').gt(BUILT_VV).unmask(0),        # walls + ground = radar double bounce
).reduce(ee.Reducer.sum()).rename('built_votes')

# --- Riverbed zone (natural sand bars) ---
big_rivers = (ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers')
              .filterBounds(aoi.bounds().buffer(5000))
              .filter(ee.Filter.gte('DIS_AV_CMS', RIVER_MIN_DISCHARGE)))
riverbed = ee.Image(0).paint(big_rivers.map(lambda f: f.buffer(RIVERBED_BUFFER, 50)), 1).rename('riverbed')

# --- AlphaEarth embeddings -> similarity to typical pits / buildings / water / crops ---
emb_col = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL').filterBounds(aoi)
emb_year = ee.Date(emb_col.filterDate('2017-01-01', end).aggregate_max('system:time_start')).get('year')
emb_start = ee.Date.fromYMD(emb_year, 1, 1)
embeddings = emb_col.filterDate(emb_start, emb_start.advance(1, 'year')).mosaic()
EMB = [f'A{i:02d}' for i in range(64)]

ndvi_p90, ndwi_p50, ndwi_p90 = stats.select('NDVI_p90'), stats.select('NDWI_p50'), stats.select('NDWI_p90')
prev_ndvi = previous.select('NDVI_prev_p90')
never_green = ndvi_p90.lt(NEVER_GREEN_NDVI)
pit_like = (prev_ndvi.gt(GREEN_NDVI).And(never_green)                                  # dug since last year
            .Or(never_green.And(prev_ndvi.lt(NEVER_GREEN_NDVI))
                .And(ndwi_p50.lte(PERMANENT_WATER_NDWI)).And(riverbed.Not()))           # older, not a sand bar
            .And(ndwi_p90.gt(WET_NDWI))
            .And(built_votes.eq(0))
            .And(radar.select('VV_p50').lt(PIT_MAX_VV)))
prototype_masks = {
    'pit': pit_like,
    'buildings': built_votes.gte(2),
    'water': ndwi_p50.gt(PERMANENT_WATER_NDWI).And(previous.select('NDWI_prev_p50').gt(PERMANENT_WATER_NDWI)),
    'crops': ndvi_p90.gt(GREEN_NDVI).And(built_votes.eq(0)),
}
# Typical (mean) embedding of each class, computed once over the test box.
prototypes = ee.Dictionary({
    name: embeddings.select(EMB).updateMask(mask).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=test_box, scale=30,
        maxPixels=1e9, bestEffort=True, tileScale=4)
    for name, mask in prototype_masks.items()}).getInfo()

def similarity_band(name):
    vector = prototypes.get(name) or {}
    if any(vector.get(b) is None for b in EMB):
        print(f'  no {name} examples in the test box: sim_{name} set to 0')
        return ee.Image.constant(0).rename(f'sim_{name}')
    v = np.array([vector[b] for b in EMB])
    v = v / np.linalg.norm(v)
    return (embeddings.select(EMB).multiply(ee.Image.constant(v.tolist()))
            .reduce(ee.Reducer.sum()).rename(f'sim_{name}'))

similarity = ee.Image.cat(*[similarity_band(n) for n in prototype_masks])

all_bands = ee.Image.cat(median, stats, previous, radar, dynamic_world,
                         building_presence, built_votes, similarity, riverbed)
stack = (ee.Image.cat(*[all_bands.select(b).multiply(f).round().toInt16() for b, f in BANDS.items()])
         .clip(aoi).unmask(NODATA, False))

info = ee.Dictionary({'s2': s2.size(), 's1': s1.size(), 'emb_year': emb_year,
                      'km2': aoi.area(100).divide(1e6)}).getInfo()
print(f"Area {info['km2']:.0f} km2 | Sentinel-2 scenes {info['s2']} | Sentinel-1 scenes {info['s1']} | "
      f"AlphaEarth year {info['emb_year']}")

## 3. Look at the inputs — and label pits (recommended)
The **Satellite** basemap is high-resolution: zoom to 16–18, use the **marker** tool to click on pits, then press
**Save drawn as PITS**. Do the same for things that are *not* pits but might look like them
(roofs, construction sites, brick kilns, natural sand bars, canals, fallow fields) with **Save drawn as NOT pits**.
30–50 of each is a good start. Points are saved to `pit_outputs/labels.geojson` and used from section 6 on.
The basemap can be older than the Sentinel window — label pits you can also see in the Sentinel-2 layer.

In [ ]:
m = geemap.Map(height='620px')
m.add_basemap('SATELLITE')
m.addLayer(median, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3, 'gamma': 1.2}, 'Sentinel-2 median', False)
m.addLayer(ndwi_p90, {'min': -0.5, 'max': 0.3, 'palette': ['f5deb3', 'ffffff', '2166ac']},
           'Wettest NDWI (pits hold water)', False)
m.addLayer(radar.select('VV_p50'), {'min': -22, 'max': -2}, 'Radar VV (bright = buildings)', False)
m.addLayer(dynamic_world.select(['dw_bare', 'dw_crops', 'dw_water']), {'min': 0, 'max': 0.8},
           'Dynamic World: R bare, G crops, B water', False)
m.addLayer(built_votes.selfMask(), {'min': 1, 'max': 4, 'palette': ['fee391', 'fe9929', 'cc4c02', '662506']},
           'Building sources agreeing (1-4)', False)
m.addLayer(similarity.select('sim_pit'), {'min': 0.6, 'max': 1, 'palette': ['000000', 'ff8c00', 'ffff00']},
           'AlphaEarth: similarity to typical pits', False)
m.addLayer(riverbed.selfMask(), {'palette': ['00bfff']}, 'Riverbed zone', False)
m.addLayer(pit_like.selfMask(), {'palette': ['ff0000']}, 'Automatic pit examples')
m.addLayer(ee.Image().byte().paint(districts, 1, 2), {'palette': ['ffff00']}, 'District')
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
           {'palette': ['00ffff']}, 'Analysis area')


def load_labels():
    return json.loads(LABELS_PATH.read_text())['features'] if LABELS_PATH.exists() else []


def save_labels(features):
    LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
    LABELS_PATH.write_text(json.dumps({'type': 'FeatureCollection', 'features': features}))


def label_summary():
    labels = load_labels()
    n_pits = sum(f['properties']['class'] == 1 for f in labels)
    return f'{n_pits} pit points and {len(labels) - n_pits} not-pit points saved in {LABELS_PATH}'


def show_labels():
    labels = load_labels()
    for value, color, name in [(1, 'ff0000', 'Your pit points'), (0, '0000ff', 'Your not-pit points')]:
        points = [ee.Feature(ee.Geometry(f['geometry'])) for f in labels if f['properties']['class'] == value]
        if points:
            m.addLayer(ee.FeatureCollection(points), {'color': color}, name)


def save_drawn(value):
    if not m.draw_features:
        status.value = 'Draw points with the marker tool first.'
        return
    drawn = ee.FeatureCollection(m.draw_features).getInfo()['features']
    save_labels(load_labels() + [{'type': 'Feature', 'geometry': f['geometry'],
                                  'properties': {'class': value}} for f in drawn])
    m.remove_drawn_features()
    show_labels()
    status.value = label_summary()


def remove_last(_):
    save_labels(load_labels()[:-1])
    status.value = label_summary()


status = widgets.Label(label_summary())
buttons = [widgets.Button(description='Save drawn as PITS', button_style='danger'),
           widgets.Button(description='Save drawn as NOT pits', button_style='info'),
           widgets.Button(description='Remove last saved point')]
buttons[0].on_click(lambda _: save_drawn(1))
buttons[1].on_click(lambda _: save_drawn(0))
buttons[2].on_click(remove_last)
show_labels()
m.centerObject(aoi, 12)
display(widgets.VBox([m, widgets.HBox(buttons), status]))

## 4. Download the features
One multi-band GeoTIFF on a 10 m UTM grid, fetched in 256 × 256 tiles in parallel.
If it stops (network, quota), just run the cell again — finished tiles are skipped.

In [ ]:
def grid_for(region):
    """10 m UTM pixel grid covering `region`: (transform, width, height)."""
    ring = region.bounds(1, CRS).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in ring], [p[1] for p in ring]
    x0 = math.floor(min(xs) / SCALE) * SCALE
    y0 = math.ceil(max(ys) / SCALE) * SCALE
    width = int(math.ceil((max(xs) - x0) / SCALE))
    height = int(math.ceil((y0 - min(ys)) / SCALE))
    return Affine(SCALE, 0, x0, 0, -SCALE, y0), width, height


def download_stack(image, region, path, tile=256, workers=12):
    transform, width, height = grid_for(region)
    tiles = [(c, r, min(tile, width - c), min(tile, height - r))
             for r in range(0, height, tile) for c in range(0, width, tile)]
    progress = Path(str(path) + '.progress.json')
    saved = json.loads(progress.read_text()) if progress.exists() and path.exists() else []
    done = [tuple(t) if len(t) == 4 else (t[0], t[1], 512, 512) for t in saved]   # older files: 512 px tiles

    def already_done(t):          # finished windows (any tile size) that contain this tile
        c, r, w, h = t
        return any(dc <= c and dr <= r and c + w <= dc + dw and r + h <= dr + dh for dc, dr, dw, dh in done)

    todo = [t for t in tiles if not already_done(t)]
    print(f'{width} x {height} pixels, {len(BANDS)} bands '
          f'(~{width * height * len(BANDS) * 2 / 1e9:.1f} GB before compression); '
          f'{len(todo)} of {len(tiles)} tiles to fetch')

    def fetch(t):
        c, r, w, h = t
        request = {
            'expression': image,
            'fileFormat': 'NUMPY_NDARRAY',
            'grid': {'dimensions': {'width': w, 'height': h},
                     'affineTransform': {'scaleX': SCALE, 'shearX': 0, 'translateX': transform.c + c * SCALE,
                                         'shearY': 0, 'scaleY': -SCALE, 'translateY': transform.f - r * SCALE},
                     'crsCode': CRS},
        }
        for attempt in range(6):
            try:
                array = ee.data.computePixels(request)
                return t, np.stack([array[b] for b in BAND_NAMES])
            except Exception:
                if attempt == 5:
                    raise
                time.sleep(2 ** attempt)

    if done:
        dst = rasterio.open(path, 'r+')
    else:
        dst = rasterio.open(path, 'w', driver='GTiff', width=width, height=height, count=len(BANDS),
                            dtype='int16', crs=CRS, transform=transform, nodata=NODATA, tiled=True,
                            blockxsize=512, blockysize=512, compress='deflate', predictor=2, BIGTIFF='IF_SAFER')
        for i, name in enumerate(BAND_NAMES, 1):
            dst.set_band_description(i, name)
    with dst, ThreadPoolExecutor(workers) as pool:
        for future in tqdm(as_completed([pool.submit(fetch, t) for t in todo]), total=len(todo),
                           desc='Downloading'):
            (c, r, w, h), data = future.result()
            dst.write(data, window=Window(c, r, w, h))
            done.append((c, r, w, h))
            progress.write_text(json.dumps(done))
    print(f'Saved {path}')


download_stack(stack, aoi, STACK_PATH)

## 5. The downloaded data

In [ ]:
def decode_band(raw, name):
    return np.where(raw == NODATA, np.nan, raw / BANDS[name]).astype('float32')


def read_bands(names, max_size=1500):
    """Decoded bands (NaN = no data), shrunk to at most max_size pixels for display."""
    with rasterio.open(STACK_PATH) as src:
        factor = max(1, math.ceil(max(src.width, src.height) / max_size))
        out_shape = (math.ceil(src.height / factor), math.ceil(src.width / factor))
        return {n: decode_band(src.read(BAND_NAMES.index(n) + 1, out_shape=out_shape,
                                        resampling=Resampling.nearest), n) for n in names}


def shrink_max(array, factor):
    """Shrink by `factor` keeping the maximum of each block (small pits stay visible)."""
    if factor == 1:
        return array
    h, w = array.shape
    padded = np.pad(array, ((0, -h % factor), (0, -w % factor)), constant_values=0)
    return padded.reshape(padded.shape[0] // factor, factor, padded.shape[1] // factor, factor).max(axis=(1, 3))


def stretch_rgb(r, g, b, limits=None):
    """RGB for display with a 2-98 % stretch; returns (image, limits) so other views can reuse the stretch."""
    rgb = np.dstack([r, g, b]).astype('float32')
    if limits is None:
        valid = np.isfinite(rgb).all(axis=2)
        limits = np.nanpercentile(rgb[valid], [2, 98]) if valid.any() else (0, 0.3)
    lo, hi = limits
    return np.clip(np.nan_to_num((rgb - lo) / max(hi - lo, 1e-6)), 0, 1), limits


view = read_bands(['B4', 'B3', 'B2', 'NDWI_p90', 'NDVI_p90', 'NDVI_prev_p90', 'VV_p50',
                   'dw_bare', 'dw_crops', 'dw_water', 'built_votes', 'sim_pit'])
sim = view['sim_pit']
panels = [
    ('Sentinel-2 true colour', stretch_rgb(view['B4'], view['B3'], view['B2'])[0], {}),
    ('Wettest NDWI (water)', view['NDWI_p90'], dict(cmap='BrBG', vmin=-0.5, vmax=0.3)),
    ('Greenest NDVI now', view['NDVI_p90'], dict(cmap='YlGn', vmin=0, vmax=0.8)),
    ('Greenest NDVI a year earlier', view['NDVI_prev_p90'], dict(cmap='YlGn', vmin=0, vmax=0.8)),
    ('Radar VV (dB)', view['VV_p50'], dict(cmap='gray', vmin=-22, vmax=-2)),
    ('Dynamic World: R bare, G crops, B water',
     np.clip(np.nan_to_num(np.dstack([view['dw_bare'], view['dw_crops'], view['dw_water']])) / 0.8, 0, 1), {}),
    ('Building sources agreeing', view['built_votes'], dict(cmap='OrRd', vmin=0, vmax=4)),
    ('AlphaEarth: similarity to pits', sim,
     dict(cmap='inferno', vmin=np.nanpercentile(sim, 5), vmax=np.nanpercentile(sim, 99.8))),
]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (title, img, kw) in zip(axes.flat, panels):
    im = ax.imshow(img, **kw)
    ax.set_title(title)
    ax.axis('off')
    if kw:
        fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inputs_overview.png', dpi=110)
plt.show()

## 6. Features at full 10 m detail, and training examples
* **Context**: each pixel minus the mean of its 130 m neighbourhood (water, crops, bare, radar, NIR, wetness) —
  a pit is a patch that differs from the fields around it.
* **Sub-pixel fractions**: every pixel is unmixed into water / bare soil / vegetation / building shares, so a pit
  that covers only part of a pixel still shows up.
* **Automatic examples** (strata): 1 pit, 2 pit look-alike at buildings, 3 permanent water, 4 buildings, 5 crops, 6 dry bare land.
* **Your points** override automatic examples within 50 m and count `USER_WEIGHT` times.

In [ ]:
CTX, AROUND, NEAR = 13, 31, 11          # window sizes in pixels: 130 m, 310 m, 110 m
HALO = AROUND // 2 + 1                  # margin read around each block for these windows
BLOCK = 1024                            # processing block size (pixels)
MEDIAN_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
FRACTIONS = ['water', 'bare', 'veg', 'built']
DEFAULT_ENDMEMBERS = {                  # reflectance B2, B3, B4, B8, B11, B12
    'water': [0.07, 0.08, 0.06, 0.04, 0.02, 0.015],
    'bare': [0.10, 0.14, 0.18, 0.24, 0.32, 0.27],
    'veg': [0.03, 0.06, 0.04, 0.38, 0.20, 0.10],
    'built': [0.10, 0.11, 0.12, 0.17, 0.21, 0.19],
}
STRATA = {1: 'pit', 2: 'lookalike', 3: 'water', 4: 'buildings', 5: 'crops', 6: 'dry_bare'}
CONTEXT = [('ctx_crops', 'dw_crops'), ('ctx_water', 'dw_water'), ('ctx_bare', 'dw_bare'),
           ('ctx_vv', 'VV_p50'), ('ctx_nir', 'B8'), ('ctx_ndwi90', 'NDWI_p90')]
ALL_FEATURES = ([n for n in BAND_NAMES if n != 'riverbed']
                + ['NDBI', 'dNDVI_p90', 'VV_VH'] + [name for name, _ in CONTEXT]
                + ['crops_around', 'built_nearby', 'nir_texture'] + [f'frac_{k}' for k in FRACTIONS])
# Bands the automatic pit rules are built from — withheld while automatic labels are used,
# so the model has to learn what a pit looks like instead of memorising the rules.
RULE_FEATURES = ['NDVI_p90', 'NDWI_p50', 'NDWI_p90', 'NDVI_prev_p90', 'NDWI_prev_p50', 'dNDVI_p90', 'ctx_ndwi90']
ENDMEMBERS = DEFAULT_ENDMEMBERS


def blocks(width, height, size=BLOCK):
    for r in range(0, height, size):
        for c in range(0, width, size):
            yield Window(c, r, min(size, width - c), min(size, height - r))


def read_with_halo(src, window):
    big = Window(window.col_off - HALO, window.row_off - HALO,
                 window.width + 2 * HALO, window.height + 2 * HALO)
    return src.read(window=big, boundless=True, fill_value=NODATA)


def core(a):
    return a[..., HALO:-HALO, HALO:-HALO]


def decode(raw):
    return {name: decode_band(raw[i], name) for i, name in enumerate(BAND_NAMES)}


def local_mean(x, size):
    """Mean over a size x size window, ignoring missing pixels."""
    valid = np.isfinite(x)
    total = ndimage.uniform_filter(np.where(valid, x, 0).astype('float64'), size, mode='nearest')
    count = ndimage.uniform_filter(valid.astype('float64'), size, mode='nearest')
    with np.errstate(invalid='ignore', divide='ignore'):
        return (total / count).astype('float32')


def unmix(b):
    """Sub-pixel shares of water / bare / vegetation / built (least squares, sum to one, 0-1)."""
    x = np.stack([b[n] for n in MEDIAN_BANDS]).reshape(6, -1)
    E = np.array([ENDMEMBERS[k] for k in FRACTIONS], dtype='float64').T      # 6 bands x 4 classes
    A = np.vstack([E, 10 * np.ones((1, E.shape[1]))])                        # strongly weighted sum-to-one row
    rhs = np.vstack([np.nan_to_num(x), 10 * np.ones((1, x.shape[1]))])
    f = np.clip(np.linalg.pinv(A) @ rhs, 0, 1)
    f /= np.maximum(f.sum(axis=0, keepdims=True), 1e-6)
    f[:, ~np.isfinite(x).all(axis=0)] = np.nan
    return {f'frac_{k}': f[i].reshape(b['B2'].shape).astype('float32') for i, k in enumerate(FRACTIONS)}


def compute_features(raw):
    """(decoded bands, features) for a block read with a HALO margin."""
    b = decode(raw)
    f = {name: b[name] for name in BAND_NAMES if name != 'riverbed'}
    with np.errstate(invalid='ignore', divide='ignore'):
        f['NDBI'] = (b['B11'] - b['B8']) / (b['B11'] + b['B8'])
    f['dNDVI_p90'] = b['NDVI_p90'] - b['NDVI_prev_p90']
    f['VV_VH'] = b['VV_p50'] - b['VH_p50']
    for name, source in CONTEXT:
        f[name] = b[source] - local_mean(b[source], CTX)
    f['crops_around'] = local_mean(b['dw_crops'], AROUND)
    f['built_nearby'] = local_mean((b['built_votes'] >= 1).astype('float32'), NEAR)
    f['nir_texture'] = np.sqrt(np.maximum(local_mean(b['B8'] ** 2, 3) - local_mean(b['B8'], 3) ** 2, 0))
    f.update(unmix(b))
    return b, f


def strata_for(b, built_nearby):
    never_green = b['NDVI_p90'] < NEVER_GREEN_NDVI
    candidate = (((b['NDVI_prev_p90'] > GREEN_NDVI) & never_green)                        # dug since last year
                 | (never_green & (b['NDVI_prev_p90'] < NEVER_GREEN_NDVI)
                    & ~(b['NDWI_p50'] > PERMANENT_WATER_NDWI) & (b['riverbed'] != 1))    # older, not a sand bar
                 ) & (b['NDWI_p90'] > WET_NDWI)
    building_like = (b['built_votes'] >= 1) | (built_nearby >= 0.2)
    s = np.zeros(never_green.shape, np.uint8)
    s[(b['NDVI_p90'] > GREEN_NDVI) & (b['built_votes'] == 0)] = 5
    s[never_green & (b['NDWI_p90'] < -0.1) & (b['built_votes'] == 0)] = 6
    s[candidate & building_like] = 2                                   # pit look-alike (construction...)
    s[b['built_votes'] >= 2] = 4                                       # confident buildings
    s[(b['NDWI_p50'] > PERMANENT_WATER_NDWI) & (b['NDWI_prev_p50'] > PERMANENT_WATER_NDWI)] = 3
    s[candidate & ~building_like & (b['VV_p50'] < PIT_MAX_VV)] = 1
    s[~np.isfinite(b['NDVI_p90']) | ~np.isfinite(b['VV_p50'])] = 0
    return s


def estimate_endmembers(src, max_blocks=40, seed=0):
    """Typical pure-pixel spectra from this area (falls back to defaults if too few)."""
    rng = np.random.default_rng(seed)
    all_blocks = list(blocks(src.width, src.height, 512))
    pools = {k: [] for k in FRACTIONS}
    for i in rng.permutation(len(all_blocks))[:max_blocks]:
        b = decode(src.read(window=all_blocks[i]))
        x = np.stack([b[n] for n in MEDIAN_BANDS])
        with np.errstate(invalid='ignore', divide='ignore'):
            ndvi = (b['B8'] - b['B4']) / (b['B8'] + b['B4'])
        pure = {'water': (b['NDWI_p50'] > 0.2) & (b['NDWI_prev_p50'] > 0.2),
                'bare': (b['NDVI_p90'] < 0.15) & (b['NDWI_p90'] < -0.15) & (b['built_votes'] == 0),
                'veg': ndvi > 0.7,
                'built': b['built_votes'] >= 3}
        for k, mask in pure.items():
            mask = mask & np.isfinite(x).all(axis=0)
            if mask.any():
                pools[k].append(x[:, mask][:, :5000])
    result = {}
    for k in FRACTIONS:
        pixels = np.concatenate(pools[k], axis=1) if pools[k] else np.empty((6, 0))
        result[k] = np.median(pixels, axis=1).tolist() if pixels.shape[1] >= 50 else DEFAULT_ENDMEMBERS[k]
        print(f"  endmember {k:5s}: {'from ' + str(pixels.shape[1]) + ' pure pixels' if pixels.shape[1] >= 50 else 'default'}")
    return result


def automatic_samples(src, seed=42):
    """Two passes: count the strata everywhere (and save strata.tif), then sample each to its target."""
    rng = np.random.default_rng(seed)
    all_blocks = list(blocks(src.width, src.height))
    counts = np.zeros(8, np.int64)
    profile = src.profile.copy()
    profile.update(count=1, dtype='uint8', nodata=0, predictor=1)
    with rasterio.open(OUT_DIR / 'strata.tif', 'w', **profile) as dst:
        for w in tqdm(all_blocks, desc='Automatic labels 1/2'):
            b = decode(read_with_halo(src, w))
            s = core(strata_for(b, local_mean((b['built_votes'] >= 1).astype('float32'), NEAR)))
            dst.write(s, 1, window=w)
            counts += np.bincount(s.ravel(), minlength=8)
    rate = np.zeros(8)
    for code, name in STRATA.items():
        rate[code] = min(1.0, SAMPLES[name] / max(counts[code], 1))
    parts = []
    with rasterio.open(OUT_DIR / 'strata.tif') as strata_src:
        for w in tqdm(all_blocks, desc='Automatic labels 2/2'):
            s = strata_src.read(1, window=w)
            pick = rng.random(s.shape) < rate[s]
            if not pick.any():
                continue
            _, f = compute_features(read_with_halo(src, w))
            rr, cc = np.nonzero(pick)
            part = pd.DataFrame({name: core(v)[rr, cc] for name, v in f.items()})
            part['row'], part['col'], part['stratum'] = rr + int(w.row_off), cc + int(w.col_off), s[rr, cc]
            parts.append(part)
    auto = pd.concat(parts, ignore_index=True)
    auto['class'] = (auto['stratum'] == 1).astype(int)
    auto['source'], auto['weight'] = 'auto', 1.0
    return auto, counts


def user_samples(src):
    """Features at your points (3 x 3 pixels each); polygons use their centre."""
    features = json.loads(LABELS_PATH.read_text())['features'] if LABELS_PATH.exists() else []
    to_grid = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
    parts = []
    for feature in features:
        x, y = to_grid.transform(*shape(feature['geometry']).centroid.coords[0])
        r, c = src.index(x, y)
        if not (1 <= r < src.height - 1 and 1 <= c < src.width - 1):
            continue
        _, f = compute_features(read_with_halo(src, Window(c - 1, r - 1, 3, 3)))
        part = pd.DataFrame({name: core(v).ravel() for name, v in f.items()})
        part['row'] = np.repeat(np.arange(r - 1, r + 2), 3)
        part['col'] = np.tile(np.arange(c - 1, c + 2), 3)
        part['class'] = int(feature['properties']['class'])
        parts.append(part)
    if not parts:
        return None
    user = pd.concat(parts, ignore_index=True)
    user['stratum'], user['source'], user['weight'] = 0, 'user', float(USER_WEIGHT)
    return user


with rasterio.open(STACK_PATH) as src:
    ENDMEMBERS = estimate_endmembers(src)
    user = user_samples(src)
    use_auto = USE_AUTO_LABELS or user is None
    auto, strata_counts = automatic_samples(src) if use_auto else (None, None)

if auto is not None and user is not None:       # your points overrule automatic examples within 50 m
    distance, _ = cKDTree(user[['row', 'col']].values).query(auto[['row', 'col']].values)
    auto = auto[distance > 5]
samples = pd.concat([part for part in (auto, user) if part is not None], ignore_index=True)
bx, by = samples['col'] // BLOCK_PX, samples['row'] // BLOCK_PX
samples['validation'] = (bx * 7919 + by * 104729) % 100 < VALIDATION_PERCENT
if samples['validation'].all() or not samples['validation'].any():
    print('Area too small for block validation: falling back to a random 25 % split (scores will be optimistic).')
    samples['validation'] = np.random.default_rng(0).random(len(samples)) < 0.25
samples.to_csv(OUT_DIR / 'training_samples.csv', index=False)

if strata_counts is not None:
    print('Pixels per automatic stratum:',
          {STRATA[c]: int(strata_counts[c]) for c in STRATA})
display(samples.groupby(['source', 'class', 'validation']).size().unstack(fill_value=0)
        .rename(columns={False: 'training', True: 'validation'}))

# Map of the strata and where the samples are
with rasterio.open(OUT_DIR / 'strata.tif' if use_auto else STACK_PATH) as src:
    factor = max(1, math.ceil(max(src.width, src.height) / 1500))
    strata_view = (src.read(1, out_shape=(math.ceil(src.height / factor), math.ceil(src.width / factor)),
                            resampling=Resampling.mode) if use_auto else None)
fig, ax = plt.subplots(figsize=(10, 9))
if strata_view is not None:
    cmap = ListedColormap(['#00000000', 'red', 'orange', 'blue', 'grey', 'green', 'tan'])
    ax.imshow(strata_view, cmap=cmap, vmin=0, vmax=6, interpolation='nearest')
for (source, value), marker, color in [(('user', 1), '*', 'red'), (('user', 0), '*', 'blue')]:
    part = samples[(samples.source == source) & (samples['class'] == value)]
    ax.scatter(part.col / factor, part.row / factor, marker=marker, s=60, c=color, edgecolors='white')
ax.set_title('Automatic strata: red pit, orange look-alike, blue water, grey buildings, green crops, tan dry bare; '
             'stars = your points', fontsize=9)
ax.axis('off')
plt.show()

## 7. Train and validate
Validation uses whole 5 km blocks the model never trained on. The table shows, for each threshold,
**precision** (share of detections that are real pits) and **recall** (share of real pits found).
Scores against *your points* are real accuracy; scores against the automatic rules only measure agreement.

In [ ]:
use_auto = (samples['source'] == 'auto').any()
FEATURES = [f for f in ALL_FEATURES if not (use_auto and f in RULE_FEATURES)]
train, valid = samples[~samples.validation], samples[samples.validation]


def make_model():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
                                          min_samples_leaf=20, l2_regularization=1.0,
                                          class_weight='balanced', random_state=42)


model = make_model().fit(train[FEATURES], train['class'], sample_weight=train['weight'])
valid = valid.assign(prob=model.predict_proba(valid[FEATURES])[:, 1])
THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.05), 2)


def threshold_table(df):
    rows = []
    for t in THRESHOLDS:
        found = df.prob >= t
        tp = int((found & (df['class'] == 1)).sum())
        fp = int((found & (df['class'] == 0)).sum())
        fn = int((~found & (df['class'] == 1)).sum())
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        rows.append({'threshold': t, 'precision': precision, 'recall': recall,
                     'f1': 2 * precision * recall / max(precision + recall, 1e-9)})
    return pd.DataFrame(rows)


tables = {}
for source, title in [('user', 'your points (real accuracy)'), ('auto', 'automatic rules (agreement only)')]:
    part = valid[valid.source == source]
    n_pit, n_other = int((part['class'] == 1).sum()), int((part['class'] == 0).sum())
    if min(n_pit, n_other) >= 10:
        tables[source] = threshold_table(part)
        print(f'Validation against {title}: {n_pit} pit / {n_other} not-pit samples')
        display(tables[source].round(3))
    elif source == 'user' and len(part):
        print(f'Only {n_pit} pit / {n_other} not-pit samples of yours in validation blocks — add more points.')

# Agreement with the automatic rules says nothing about real accuracy, so only your points tune the threshold.
if PROB_HIGH == 'auto' and 'user' in tables:
    reference = tables['user']
    best = reference[reference.f1 >= reference.f1.max() - 0.01]              # ties: prefer the middle
    prob_high, basis = float(best.loc[(best.threshold - 0.5).abs().idxmin(), 'threshold']), 'best F1 on your points'
elif PROB_HIGH == 'auto':
    prob_high, basis = 0.6, 'default — add your points in section 3 to tune it'
else:
    prob_high, basis = float(PROB_HIGH), 'setting'
prob_low = max(0.2, prob_high - 0.25)
print(f'Pit threshold {prob_high:.2f} ({basis}); outlines grow over connected pixels >= {prob_low:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for source, table in tables.items():
    axes[0].plot(table.recall, table.precision, marker='o', label=source)
    for _, r in table.iloc[::3].iterrows():
        axes[0].annotate(f'{r.threshold:.2f}', (r.recall, r.precision), fontsize=8)
axes[0].set(xlabel='Recall (real pits found)', ylabel='Precision (detections that are real)',
            title='Precision vs recall on validation blocks', xlim=(0, 1.02), ylim=(0, 1.02))
axes[0].legend()
subset = valid.sample(min(len(valid), 3000), random_state=0)
if subset['class'].nunique() == 2:
    importance = permutation_importance(model, subset[FEATURES], subset['class'], scoring='average_precision',
                                        n_repeats=3, random_state=0, n_jobs=-1)
    pd.Series(importance.importances_mean, index=FEATURES).sort_values().tail(15).plot.barh(ax=axes[1], color='#d95f02')
    axes[1].set_title('What the model relies on (permutation importance)')
plt.tight_layout()
plt.show()

# Final model: all samples (training + validation).
final_model = make_model().fit(samples[FEATURES], samples['class'], sample_weight=samples['weight'])

## 8. Pit probability for every 10 m pixel

In [ ]:
with rasterio.open(STACK_PATH) as src:
    profile = src.profile.copy()
    profile.update(count=1, dtype='uint8', nodata=255, predictor=1)
    with rasterio.open(PROB_PATH, 'w', **profile) as dst:
        for w in tqdm(list(blocks(src.width, src.height)), desc='Predicting'):
            b, f = compute_features(read_with_halo(src, w))
            valid_px = core(np.isfinite(b['NDVI_p90']) & np.isfinite(b['VV_p50']))
            out = np.full((int(w.height), int(w.width)), 255, np.uint8)
            if valid_px.any():
                X = pd.DataFrame({name: core(f[name])[valid_px] for name in FEATURES})
                out[valid_px] = np.round(final_model.predict_proba(X)[:, 1] * 100).astype(np.uint8)
            dst.write(out, 1, window=w)

with rasterio.open(PROB_PATH) as src:
    factor = max(1, math.ceil(max(src.width, src.height) / 1500))
    full = src.read(1)
shown = shrink_max(np.where(full == 255, 0, full), factor)        # block maximum keeps small pits visible
shown = np.ma.masked_where(shown < prob_low * 100, shown / 100)
overview_rgb, _ = stretch_rgb(view['B4'], view['B3'], view['B2'])
fig, ax = plt.subplots(figsize=(11, 10))
ax.imshow(overview_rgb)
im = ax.imshow(shown, cmap='magma', vmin=prob_low, vmax=1, alpha=0.85)
fig.colorbar(im, ax=ax, fraction=0.035, label='pit probability')
ax.set_title(f'Pit probability (>= {prob_low:.2f}) over Sentinel-2')
ax.axis('off')
plt.show()
print(f'Saved {PROB_PATH}')

## 9. Pit objects and outlines
* **Hysteresis**: a pit needs at least one pixel above the threshold, then grows over connected pixels above the lower threshold — tighter outlines, fewer specks.
* Pixels where 2+ building sources agree are removed.
* **Tiny pits** (under 10 pixels ≈ 0.1 ha) are kept only when very confident and not line-shaped; **long thin** shapes (canals, roads, tracks, drains) and outlines that are mostly building are dropped.
* Every pit is tagged **Yamuna riverbed** or **outside riverbed**: riverbed detections can be sand mining, but also natural sand bars, flood pools or bank erosion — review them separately.

In [ ]:
with rasterio.open(PROB_PATH) as src:
    prob_raw, grid_transform, grid_crs = src.read(1), src.transform, src.crs
with rasterio.open(STACK_PATH) as src:
    def full_band(name):
        return decode_band(src.read(BAND_NAMES.index(name) + 1), name)
    votes, ndwi90, river = full_band('built_votes'), full_band('NDWI_p90'), full_band('riverbed')
    ndwi_p50 = full_band('NDWI_p50')
    ndvi90, ndvi_prev = full_band('NDVI_p90'), full_band('NDVI_prev_p90')

prob = np.where(prob_raw == 255, 0, prob_raw / 100).astype('float32')
confident_built = votes >= 2
pit_pixels = apply_hysteresis_threshold(prob, prob_low, prob_high) & ~confident_built
removed_as_building = int(((prob >= prob_high) & confident_built).sum())
labels = measure.label(pit_pixels, connectivity=2).astype(np.int32)
ids = np.arange(1, labels.max() + 1)

# Linear features: long, narrow strips that are pit-like or wet most of the time (canals, drains, roads).
strips = (prob >= 0.15) | (ndwi_p50 > PERMANENT_WATER_NDWI)
strip_labels, n_strips = ndimage.label(strips, structure=np.ones((3, 3)))
linear = np.zeros(prob.shape, bool)
if n_strips:
    strip_ids = np.arange(1, n_strips + 1)
    half_width = ndimage.maximum(ndimage.distance_transform_edt(strips), strip_labels, strip_ids)
    length = np.array([math.hypot(sl[0].stop - sl[0].start, sl[1].stop - sl[1].start)
                       for sl in ndimage.find_objects(strip_labels)])
    linear = np.isin(strip_labels, strip_ids[(np.asarray(half_width) <= LINEAR_MAX_HALF_WIDTH)
                                             & (length >= LINEAR_MIN_LENGTH)])
del strip_labels
# Share of buildings within ~100 m of each pixel (for village ponds).
near_buildings = local_mean((votes >= 2).astype('float32'), 21)

props = pd.DataFrame(measure.regionprops_table(
    labels, properties=['label', 'area', 'perimeter', 'centroid', 'axis_major_length', 'axis_minor_length']))
props = props.rename(columns={'area': 'pixels', 'centroid-0': 'row', 'centroid-1': 'col'})
if len(ids):
    props['prob_max'] = ndimage.maximum(prob, labels, ids)
    props['prob_mean'] = ndimage.mean(prob, labels, ids)
    props['water_share'] = ndimage.mean((ndwi90 > WET_NDWI).astype('float32'), labels, ids)
    props['new_share'] = ndimage.mean(((ndvi_prev > GREEN_NDVI) & (ndvi90 < NEVER_GREEN_NDVI)).astype('float32'),
                                      labels, ids)
    props['built_share'] = ndimage.mean((votes >= 1).astype('float32'), labels, ids)
    props['riverbed_share'] = ndimage.mean((river == 1).astype('float32'), labels, ids)
    props['linear_share'] = ndimage.mean(linear.astype('float32'), labels, ids)
    props['buildings_near'] = ndimage.mean(near_buildings, labels, ids)
    props['compactness'] = np.minimum(4 * np.pi * props.pixels / np.maximum(props.perimeter, 1) ** 2, 1)
    props['elongation'] = props.axis_major_length / np.maximum(props.axis_minor_length, 1)

    tiny = props.pixels < SMALL_PIT_PIXELS
    checks = {
        'too small': props.pixels < MIN_PIT_PIXELS,
        'tiny and not confident': tiny & (props.prob_max < SMALL_PIT_MIN_PROB),
        'tiny and line-shaped (track / drain)': tiny & (props.elongation > MAX_TINY_ELONGATION),
        'thin line (track / drain / road)': (props.pixels < 50) & (props.elongation > MAX_SMALL_ELONGATION),
        'mostly building': props.built_share > MAX_BUILT_SHARE,
        'long and thin (canal / road)': ~tiny & (props.compactness < MIN_COMPACTNESS),
        'part of a canal / drain / road': props.linear_share >= 0.5,
    }
    drop = np.zeros(len(props), bool)
    for reason, mask in checks.items():
        print(f'  dropped, {reason}: {int((mask & ~drop).sum())}')
        drop |= mask.values
    kept = props[~drop].copy()
else:
    kept = props.assign(prob_max=[], prob_mean=[], water_share=[], new_share=[], built_share=[],
                        riverbed_share=[], compactness=[], elongation=[], linear_share=[],
                        buildings_near=[])
print(f'  pixels removed as buildings: {removed_as_building}')

outlines = {}
for geometry, value in rio_features.shapes(labels, mask=np.isin(labels, kept.label.values),
                                           transform=grid_transform, connectivity=8):
    outlines.setdefault(int(value), []).append(shape(geometry))
pits = gpd.GeoDataFrame(kept, geometry=[unary_union(outlines[int(i)]) for i in kept.label], crs=grid_crs)
pits['area_m2'] = pits.pixels * SCALE ** 2
pits['area_ha'] = pits.area_m2 / 1e4
pits['new_pit'] = pits.new_share >= 0.5
# Riverbed detections can be sand mining or natural sand bars / flood pools — review them separately.
pits['setting'] = np.where(pits.riverbed_share >= 0.5, 'Yamuna riverbed', 'outside riverbed')
# Water bodies right next to villages are usually village ponds (johads), not mining pits.
pits['category'] = np.where((pits.buildings_near >= VILLAGE_POND_BUILT) & (pits.water_share >= 0.6) & ~pits.new_pit,
                        'likely village pond', 'pit')
# Location of each pit in degrees (WGS84): its centre, or a point inside it when the shape is curved.
inside = [g.centroid if g.contains(g.centroid) else g.representative_point() for g in pits.geometry]
centres = gpd.GeoSeries(inside, crs=grid_crs).to_crs(4326)
pits['latitude'], pits['longitude'] = centres.y.round(6).values, centres.x.round(6).values
pits['google_maps'] = [f'https://www.google.com/maps?q={la:.6f},{lo:.6f}'
                       for la, lo in zip(pits.latitude, pits.longitude)]
pits['size_class'] = pd.cut(pits.area_ha, [0, 0.1, 0.5, 2, np.inf], right=False,
                            labels=['tiny (<0.1 ha)', 'small (0.1-0.5 ha)', 'medium (0.5-2 ha)', 'large (>2 ha)'])
pits = pits.sort_values('prob_max', ascending=False).reset_index(drop=True)
pits.insert(0, 'pit_id', np.arange(1, len(pits) + 1))


def write_kml(gdf, path):
    """KML for Google Earth: outline + labelled pin per pit; red = new, magenta = older, blue = village pond."""
    marks = []
    for _, p in gdf.iterrows():
        polygons = [p.geometry] if p.geometry.geom_type == 'Polygon' else list(p.geometry.geoms)
        rings = ''.join('<Polygon><outerBoundaryIs><LinearRing><coordinates>'
                        + ' '.join(f'{x:.6f},{y:.6f}' for x, y in poly.exterior.coords)
                        + '</coordinates></LinearRing></outerBoundaryIs></Polygon>' for poly in polygons)
        pin = f'<Point><coordinates>{p.longitude:.6f},{p.latitude:.6f}</coordinates></Point>'
        about = (f"Latitude {p.latitude:.6f}, longitude {p.longitude:.6f}<br>{p.area_ha:.2f} ha, probability "
                 f"{p.prob_max:.2f}, {'new' if p.new_pit else 'older'}, {p.setting}, {p.category}<br>"
                 f'<a href="{p.google_maps}">Open in Google Maps</a>')
        marks.append(f"<Placemark><name>Pit {p.pit_id}</name><description><![CDATA[{about}]]></description>"
                     f"<styleUrl>#{'pond' if p.category != 'pit' else 'new' if p.new_pit else 'old'}</styleUrl>"
                     f"<MultiGeometry>{pin}{rings}</MultiGeometry></Placemark>")
    style = ('<Style id="{0}"><IconStyle><color>{1}</color><scale>0.6</scale></IconStyle>'
             '<LabelStyle><scale>0.7</scale></LabelStyle><LineStyle><color>{1}</color><width>2</width></LineStyle>'
             '<PolyStyle><color>{2}</color></PolyStyle></Style>')
    path.write_text('<?xml version="1.0" encoding="UTF-8"?><kml xmlns="http://www.opengis.net/kml/2.2"><Document>'
                    + style.format('new', 'ff2020ff', '402020ff') + style.format('old', 'ffff00ff', '40ff00ff')
                    + style.format('pond', 'ffffaa00', '40ffaa00')
                    + ''.join(marks) + '</Document></kml>')


columns = ['pit_id', 'latitude', 'longitude', 'category', 'area_ha', 'area_m2', 'prob_max', 'prob_mean', 'new_pit',
           'setting', 'water_share', 'buildings_near', 'compactness', 'size_class', 'google_maps']
export = (pits[columns + ['geometry']]
          .assign(size_class=pits.size_class.astype(str), new_pit=pits.new_pit.astype(int))
          .round({'area_ha': 3, 'prob_max': 3, 'prob_mean': 3, 'water_share': 3, 'buildings_near': 3,
                  'compactness': 3}))
if len(export):
    export.to_file(OUT_DIR / 'pits.gpkg', driver='GPKG')
    export.to_crs(4326).to_file(OUT_DIR / 'pits.geojson', driver='GeoJSON')
    write_kml(export.to_crs(4326), OUT_DIR / 'pits.kml')
export.drop(columns='geometry').to_csv(OUT_DIR / 'pits.csv', index=False)

only_pits = pits[pits.category == 'pit']
print(f"\n{len(only_pits)} pits, {only_pits.area_ha.sum():.1f} ha in total; {int(only_pits.new_pit.sum())} new since "
      f"this time last year (+ {int((pits.category != 'pit').sum())} likely village ponds, listed separately)")
display(only_pits.groupby('size_class', observed=False).agg(pits=('pit_id', 'size'), hectares=('area_ha', 'sum'),
                                                            new=('new_pit', 'sum')))
display(only_pits.groupby('setting').agg(pits=('pit_id', 'size'), hectares=('area_ha', 'sum'), new=('new_pit', 'sum')))
print('Most confident pits, with coordinates:')
display(pits[columns].drop(columns='google_maps').head(15))

## 10. Super-resolution to 2.5 m (optional)
[SEN2SR](https://github.com/ESAOpenSR/SEN2SR) (ESA OpenSR, MIT licence) turns Sentinel-2 10 m red/green/blue/NIR into 2.5 m.
It is used for close-up views and to tighten pit outlines in the gallery — it sharpens what Sentinel saw, it cannot add
detail that was never recorded. Needs `pip install sen2sr mlstac torch`; runs on CPU (a GPU is faster).

In [ ]:
sr_model, sr_device = None, None
if USE_SUPER_RESOLUTION:
    try:
        import torch
        import mlstac
        model_dir = Path('pit_outputs') / 'models' / 'SEN2SRLite_RGBN'
        if not model_dir.exists() or not any(model_dir.iterdir()):
            mlstac.download(file='https://huggingface.co/tacofoundation/sen2sr/resolve/main/'
                                 'SEN2SRLite/NonReference_RGBN_x4/mlm.json',
                            output_dir=str(model_dir))
        sr_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        sr_model = mlstac.load(str(model_dir)).compiled_model(device=sr_device)
        print(f'Super-resolution model ready on {sr_device}.')
    except ImportError:
        print('Super-resolution skipped: run  %pip install sen2sr mlstac torch  and re-run this cell.')


def super_resolve(rgbn):
    """(4, H, W) reflectance in the order B4, B3, B2, B8 at 10 m -> (4, 4H, 4W) at 2.5 m."""
    import torch
    x = torch.from_numpy(np.nan_to_num(rgbn).astype('float32')).to(sr_device)
    with torch.no_grad():
        return sr_model(x[None]).squeeze(0).float().cpu().numpy()


def refine_outline(sr, pit_mask):
    """2.5 m outline: water or bare super-resolved pixels within the (slightly grown) 10 m outline."""
    red, green, nir = sr[0], sr[1], sr[3]
    with np.errstate(invalid='ignore', divide='ignore'):
        ndwi, ndvi = (green - nir) / (green + nir), (nir - red) / (nir + red)
    up = np.kron(pit_mask, np.ones((4, 4), bool))
    refined = ndimage.binary_dilation(up, iterations=2) & ((ndwi > 0) | (ndvi < 0.2))
    parts, n = ndimage.label(refined)
    if n:
        touching = np.nonzero(ndimage.sum(up, parts, range(1, n + 1)) > 0)[0] + 1
        refined = np.isin(parts, touching)
    return refined

## 11. Gallery: the most confident pits and the smallest ones
Each row: Sentinel-2 at 10 m with the detected outline (yellow) · super-resolved 2.5 m with the refined outline (cyan) ·
wettest NDWI · radar · pit probability. Pages are saved as PNG.

In [ ]:
SR_CHIP = 128                               # super-resolution input: 1.28 km (the model's training size)


def view_size(pixels):
    """Close-up for small pits, wider view for big ones: 240 m to 640 m."""
    return int(np.clip(8 * math.sqrt(pixels), 24, 64))


def crop(array, row, col, size, fill):
    out = np.full((size, size), fill, array.dtype)
    r0, c0 = row - size // 2, col - size // 2
    rs, cs, re, ce = max(r0, 0), max(c0, 0), min(r0 + size, array.shape[0]), min(c0 + size, array.shape[1])
    if re > rs and ce > cs:
        out[rs - r0:re - r0, cs - c0:ce - c0] = array[rs:re, cs:ce]
    return out


def read_chip(src, names, row, col, size):
    window = Window(col - size // 2, row - size // 2, size, size)
    raw = src.read([BAND_NAMES.index(n) + 1 for n in names], window=window, boundless=True, fill_value=NODATA)
    return {n: decode_band(raw[i], n) for i, n in enumerate(names)}


half = GALLERY_SIZE // 2
gallery_pool = pits[pits.category == 'pit']
chosen = pd.concat([gallery_pool.head(half), gallery_pool[gallery_pool.area_ha < 0.1].head(half)]).drop_duplicates('pit_id')
refined_area = {}
names = ['B4', 'B3', 'B2', 'B8', 'NDWI_p90', 'VV_p50']
n_columns = 5 if sr_model is not None else 4
with rasterio.open(STACK_PATH) as src:
    for page, first in enumerate(range(0, len(chosen), 6), 1):
        rows = chosen.iloc[first:first + 6]
        fig, axes = plt.subplots(len(rows), n_columns, figsize=(4 * n_columns, 4.5 * len(rows)), squeeze=False)
        for axrow, (_, p) in zip(axes, rows.iterrows()):
            r, c, size = int(round(p.row)), int(round(p.col)), view_size(p.pixels)
            a = read_chip(src, names, r, c, size)
            mask = crop(labels, r, c, size, 0) == p.label
            chip_rgb, limits = stretch_rgb(a['B4'], a['B3'], a['B2'])
            header = (f"Pit {p.pit_id} · {p.latitude:.5f}° N, {p.longitude:.5f}° E\n"
                      f"{p.area_ha:.2f} ha · p={p.prob_max:.2f} · {'NEW' if p.new_pit else 'older'} · {p.setting}\n")
            panels = [(header + f'Sentinel-2 10 m ({size * SCALE} m view)', chip_rgb, {}, mask, 'yellow')]
            if sr_model is not None:
                big = read_chip(src, ['B4', 'B3', 'B2', 'B8'], r, c, SR_CHIP)
                sr = super_resolve(np.stack([big['B4'], big['B3'], big['B2'], big['B8']]))
                lo = (SR_CHIP - size) // 2 * 4
                sr = sr[:, lo:lo + size * 4, lo:lo + size * 4]            # same view as the 10 m panel
                refined = refine_outline(sr, mask)
                refined_area[p.pit_id] = refined.sum() * (SCALE / 4) ** 2
                panels.append((f'Super-resolved 2.5 m (AI estimate)\n{refined_area[p.pit_id] / 1e4:.2f} ha refined',
                               stretch_rgb(sr[0], sr[1], sr[2], limits)[0], {}, refined, 'cyan'))
            panels += [('Wettest NDWI', a['NDWI_p90'], dict(cmap='BrBG', vmin=-0.5, vmax=0.3), mask, 'black'),
                       ('Radar VV (dB)', a['VV_p50'], dict(cmap='gray', vmin=-22, vmax=-2), mask, 'yellow'),
                       ('Pit probability', crop(prob, r, c, size, 0.0), dict(cmap='magma', vmin=0, vmax=1),
                        mask, 'cyan')]
            for ax, (title, img, kw, outline, color) in zip(axrow, panels):
                ax.imshow(img, interpolation='nearest', **kw)
                if outline.any():
                    ax.contour(outline, levels=[0.5], colors=color, linewidths=1.5)
                ax.set_title(title, fontsize=9)
                ax.axis('off')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'gallery_{page}.png', dpi=100)
        plt.show()
if refined_area:
    pits['area_ha_2p5m'] = pits.pit_id.map(refined_area) / 1e4
    pits[['pit_id', 'area_ha', 'area_ha_2p5m']].dropna().to_csv(OUT_DIR / 'gallery_refined_areas.csv', index=False)

## 12. Interactive map of the results
Pit outlines (red = new since this time last year, magenta = older) and the probability layer over the Satellite basemap.
Hover a pit for its details.

In [ ]:
def probability_overlay(path, max_size=4096):
    """Probability as a transparent PNG in Web Mercator + its lat/lon bounds, for an ipyleaflet ImageOverlay."""
    with rasterio.open(path) as src:
        transform, width, height = calculate_default_transform(src.crs, 'EPSG:3857', src.width, src.height,
                                                               *src.bounds)
        factor = max(1, math.ceil(max(width, height) / max_size))
        width, height = math.ceil(width / factor), math.ceil(height / factor)
        transform = transform * Affine.scale(factor)
        mercator = np.full((height, width), 255, np.uint8)
        reproject(rasterio.band(src, 1), mercator, dst_transform=transform, dst_crs='EPSG:3857',
                  resampling=Resampling.max, src_nodata=255, dst_nodata=255)
    p = np.where(mercator == 255, 0, mercator / 100)
    rgba = (colormaps['magma'](p) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(p >= prob_low, 190, 0)
    buffer = io.BytesIO()
    Image.fromarray(rgba).save(buffer, format='PNG')
    to_wgs = Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)
    west, north = to_wgs.transform(transform.c, transform.f)
    east, south = to_wgs.transform(transform.c + width * transform.a, transform.f + height * transform.e)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode(), ((south, west), (north, east))

In [ ]:
import ipyleaflet

url, bounds = probability_overlay(PROB_PATH)
results_map = geemap.Map(height='650px')
results_map.add_basemap('SATELLITE')
results_map.addLayer(median, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3, 'gamma': 1.2},
                     'Sentinel-2 median', False)
results_map.add_layer(ipyleaflet.ImageOverlay(url=url, bounds=bounds, name='Pit probability', opacity=0.8))
shown_pits = (pits.to_crs(4326)[['pit_id', 'latitude', 'longitude', 'category', 'area_ha', 'prob_max', 'new_pit',
                                  'setting', 'size_class', 'geometry']]
              .assign(size_class=lambda d: d.size_class.astype(str), area_ha=lambda d: d.area_ha.round(3),
                      prob_max=lambda d: d.prob_max.round(2)))
ponds = shown_pits[shown_pits.category != 'pit']
if len(ponds):
    results_map.add_gdf(ponds, layer_name='Likely village ponds', zoom_to_layer=False,
                        style={'color': '#00aaff', 'weight': 1, 'fillColor': '#00aaff', 'fillOpacity': 0.2})
for is_new, name, color in [(False, 'Older pits', '#ff00ff'), (True, 'New pits', '#ff2020')]:
    part = shown_pits[(shown_pits.new_pit == is_new) & (shown_pits.category == 'pit')]
    if len(part):
        results_map.add_gdf(part, layer_name=name, zoom_to_layer=False,
                            style={'color': color, 'weight': 2, 'fillColor': color, 'fillOpacity': 0.2})
results_map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
                     {'palette': ['00ffff']}, 'Analysis area')
results_map.centerObject(aoi, 13)
results_map

## 13. When was a pit dug?
Two years of Sentinel-2 and radar at one pit. A drop in NDVI (fields gone) and a rise in NDWI (water) or fall in
radar (smooth sand / water) shows when digging started. Set `PIT_ID` to any pit from the tables above.

In [ ]:
PIT_ID = None      # None = the most confident new pit (or the most confident pit)

real = pits[pits.category == 'pit']
candidates = pits[pits.pit_id == PIT_ID] if PIT_ID else (real[real.new_pit] if real.new_pit.any() else real)
pit = candidates.iloc[0]
spot = ee.Geometry.Point([float(pit.longitude), float(pit.latitude)]).buffer(15)
history_start = end.advance(-2, 'year')


def series(collection, bands):
    fc = collection.select(bands).map(
        lambda img: ee.Feature(None, img.reduceRegion(ee.Reducer.mean(), spot, 10)).set('time', img.date().millis()))
    rows = [f['properties'] for f in fc.filter(ee.Filter.notNull(bands[:1])).getInfo()['features']]
    return pd.DataFrame(rows).assign(time=lambda d: pd.to_datetime(d.time, unit='ms')).sort_values('time')


optical = series(sentinel2(history_start, end).filterBounds(spot), ['NDVI', 'NDWI'])
radar_series = series(ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(spot).filterDate(history_start, end)
                      .filter(ee.Filter.eq('instrumentMode', 'IW'))
                      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')), ['VV'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(optical.time, optical.NDVI, 'g.-', label='NDVI (vegetation)')
ax.plot(optical.time, optical.NDWI, 'b.-', label='NDWI (water)')
ax.axvspan(pd.Timestamp(START), pd.Timestamp(END), color='orange', alpha=0.15, label='analysis window')
ax.set_ylabel('index')
ax2 = ax.twinx()
ax2.plot(radar_series.time, radar_series.VV, 'k.', alpha=0.5, label='Radar VV (dB)')
ax2.set_ylabel('radar VV (dB)')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.set_title(f"Pit {pit.pit_id}: {pit.area_ha:.2f} ha at {pit.latitude:.5f}° N, {pit.longitude:.5f}° E "
             f"({'new' if pit.new_pit else 'older'})")
plt.show()

## 14. Statistics

In [ ]:
stats_pits = pits[pits.category == 'pit']
if len(stats_pits):
    pits_all, pits = pits, stats_pits                  # statistics cover pits, not village ponds
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    bins = np.logspace(np.log10(0.01), np.log10(max(pits.area_ha.max(), 0.2)), 30)
    axes[0].hist(pits.area_ha, bins=bins, color='#8c2981')
    axes[0].set(xscale='log', xlabel='pit area (ha)', ylabel='pits', title='Pit sizes')
    counts = pits.groupby(['size_class', 'new_pit'], observed=False).size().unstack(fill_value=0)
    counts = counts.rename(columns={False: 'older', True: 'new'})
    counts.plot.bar(ax=axes[1], color={'older': '#ff00ff', 'new': '#ff2020'}, rot=20)
    axes[1].set(title='Pits by size', ylabel='pits', xlabel='')
    axes[2].imshow(overview_rgb)
    with rasterio.open(PROB_PATH) as src:
        factor = max(1, math.ceil(max(src.width, src.height) / 1500))
    axes[2].scatter(pits.col / factor, pits.row / factor, s=np.clip(pits.area_ha * 60, 6, 200),
                    c=np.where(pits.new_pit, '#ff2020', '#ff00ff'), edgecolors='white', linewidths=0.4)
    axes[2].set_title('Where the pits are (size = area, red = new)')
    # Latitude / longitude ticks: degrees converted to this map's pixels along its middle.
    to_grid, to_pixel = Transformer.from_crs('EPSG:4326', grid_crs, always_xy=True), ~grid_transform
    step = 0.1 if pits.longitude.max() - pits.longitude.min() > 0.3 else 0.05
    lon_ticks = np.arange(math.ceil(pits.longitude.min() / step) * step, pits.longitude.max(), step)
    lat_ticks = np.arange(math.ceil(pits.latitude.min() / step) * step, pits.latitude.max(), step)
    mid_lat, mid_lon = pits.latitude.median(), pits.longitude.median()
    height, width = overview_rgb.shape[:2]
    axes[2].set_xticks([(to_pixel * to_grid.transform(lo, mid_lat))[0] / factor for lo in lon_ticks],
                       [f'{lo:.2f}°E' for lo in lon_ticks], fontsize=8)
    axes[2].set_yticks([(to_pixel * to_grid.transform(mid_lon, la))[1] / factor for la in lat_ticks],
                       [f'{la:.2f}°N' for la in lat_ticks], fontsize=8)
    axes[2].set_xlim(0, width)
    axes[2].set_ylim(height, 0)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'statistics.png', dpi=110)
    plt.show()
    pits = pits_all
print(f'All outputs are in {OUT_DIR.resolve()}')
for path in sorted(OUT_DIR.glob('*')):
    if path.is_file():
        print(f'  {path.name:32s} {path.stat().st_size / 1e6:8.1f} MB')

## Improving the results
1. Look at the gallery and the results map. Where it is wrong, go back to **section 3** and save points:
   missed pits as *pits*, false alarms as *not pits*.
2. Re-run from **section 6**. The validation table now includes *your points* — the real accuracy — and the threshold is chosen from them.
3. When the test area looks right, set `AREA = 'district'`, re-run everything, and keep adding points where needed.
4. With 150+ points of each kind you can set `USE_AUTO_LABELS = False` to learn only from your judgement.

**Outputs** (in `pit_outputs/<run>/`): `pits.gpkg` / `pits.geojson` (GIS), `pits.kml` (Google Earth),
`pits.csv` (list with lat/lon, area, probability, new/older), `pit_probability.tif`, `stack.tif` (all downloaded features),
`strata.tif`, `training_samples.csv`, gallery and overview PNGs.